# 📊 Notebook 6: Base vs Fine-Tuned Model Comparison
> **Run this on Colab/Kaggle (Cloud GPU)**
We evaluate `Qwen2.5-3B-Instruct` (Base) against your Gujarati-Healthcare LoRA using SacreBLEU and ROUGE.

In [ ]:
!pip install -q sacrebleu rouge-score evaluate peft

In [ ]:
import json
import torch
import evaluate
import random
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

bleu_metric = evaluate.load("sacrebleu")
rouge_metric = evaluate.load("rouge")

def load_test_data(file_path="data/test.jsonl", num_samples=15):
    test_cases = []
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            data = json.loads(line)
            user_msg = next(m["content"] for m in data["messages"] if m["role"] == "user")
            asst_msg = next(m["content"] for m in data["messages"] if m["role"] == "assistant")
            test_cases.append((user_msg, asst_msg))
    random.seed(42)
    return random.sample(test_cases, min(num_samples, len(test_cases)))

test_samples = load_test_data()
print(f"Loaded {len(test_samples)} random samples for evaluation.")

### 1. Test BASE Model

In [ ]:
base_id = "Qwen/Qwen2.5-3B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(base_id)
base_model = AutoModelForCausalLM.from_pretrained(base_id, device_map="auto", torch_dtype=torch.float16)

predictions_base, references = [], []
print("🤖 Generating BASE MODEL Benchmarks...")
for instruction, target in tqdm(test_samples):
    messages = [{"role": "system", "content": "You are a safe, accurate Gujarati healthcare assistant."}, {"role": "user", "content": instruction}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([prompt], return_tensors="pt").to(base_model.device)
    
    with torch.no_grad():
        outputs = base_model.generate(**inputs, max_new_tokens=100, do_sample=False, pad_token_id=tokenizer.eos_token_id)
        
    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    predictions_base.append(response)
    references.append([target])

base_bleu = bleu_metric.compute(predictions=predictions_base, references=references)
base_rouge = rouge_metric.compute(predictions=predictions_base, references=references)


### 2. Test FINE-TUNED (LoRA) Model

In [ ]:
adapter_path = "models/qwen_gu_health_lora"
print("⚡ Attaching LoRA Adapters...")
ft_model = PeftModel.from_pretrained(base_model, adapter_path)

predictions_ft = []
print("🤖 Generating FINE-TUNED Benchmarks...")
for instruction, target in tqdm(test_samples):
    messages = [{"role": "system", "content": "You are a safe, accurate Gujarati healthcare assistant."}, {"role": "user", "content": instruction}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([prompt], return_tensors="pt").to(ft_model.device)
    
    with torch.no_grad():
        outputs = ft_model.generate(**inputs, max_new_tokens=100, do_sample=False, pad_token_id=tokenizer.eos_token_id)
        
    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    predictions_ft.append(response)

ft_bleu = bleu_metric.compute(predictions=predictions_ft, references=references)
ft_rouge = rouge_metric.compute(predictions=predictions_ft, references=references)


### 3. Comparison Report

In [ ]:
print("="*60)
print("🎯 ACADEMIC EVALUATION REPORT (COMPARISON) 🎯")
print("="*60)
print(f"{str('Metric').ljust(15)} | {str('Base Model').ljust(15)} | {str('Fine-Tuned (LoRA)')}")
print("-"*60)
print(f"{str('SacreBLEU').ljust(15)} | {str(round(base_bleu['score'], 2)).ljust(15)} | {str(round(ft_bleu['score'], 2))}")
print(f"{str('ROUGE-1').ljust(15)} | {str(round(base_rouge['rouge1']*100, 2)).ljust(15)} | {str(round(ft_rouge['rouge1']*100, 2))}")
print(f"{str('ROUGE-2').ljust(15)} | {str(round(base_rouge['rouge2']*100, 2)).ljust(15)} | {str(round(ft_rouge['rouge2']*100, 2))}")
print(f"{str('ROUGE-L').ljust(15)} | {str(round(base_rouge['rougeL']*100, 2)).ljust(15)} | {str(round(ft_rouge['rougeL']*100, 2))}")
print("="*60)
